In [1]:
from frust.stepper import Stepper
from pathlib import Path
from rdkit import Chem
from frust.embedder import embed_mols
import frust.vis as vis
from tooltoad.chemutils import xyz2mol

# g-xTB

In [7]:
NAME = "test"
smi = "CC1(C)C(C)(C)OB(C2=CC=CN2)O1"

mol = Chem.MolFromSmiles(smi)
mols = {NAME: mol}

embedded = embed_mols(mols, n_confs=1)

step = Stepper(
    step_type="none",
    save_output_dir=True,
    output_base=str(NAME),
    n_cores=1,
    memory_gb=30,
)

df = step.build_initial_df(embedded)
df = step.gxtb(df, options={"sp": None})
df = step.xtb(df)

2026-05-06 21:59:23 INFO  frust.stepper.NONE.job1111: Working dir: .
2026-05-06 21:59:23 INFO  frust.stepper.NONE.job1111: [gxtb] row 0 (test)…
2026-05-06 21:59:23 INFO  frust.stepper.NONE.job1111: [xtb-gfn] row 0 (test)…


In [9]:
for name, meta in df.attrs.get("frust_steps", {}).items():
    print(name, meta)

gxtb {'engine': 'gxtb', 'columns': ['gxtb-EE', 'gxtb-NT'], 'options': {'sp': None}}
xtb-gfn {'engine': 'xtb', 'columns': ['xtb-gfn-EE', 'xtb-gfn-NT'], 'options': {'gfn': 0}}


In [ ]:
f = Path("../structures/ts2.xyz")
mols = {}
with open(f, "r") as file:
    xyz_block = file.read()
    mol = xyz2mol(xyz_block)
    mols[f.stem] = (mol, [0])

vis.MolTo3DGrid(f, show_charges=True)

step = Stepper(
    step_type="none",
    save_output_dir=True,
    output_base=f"{NAME}_ts4",
    n_cores=10,
    memory_gb=30,
)
dfg = step.build_initial_dfg(mols)
dfg = step.gxtb(dfg, name="gxtb-hess", options={"hess": None}, n_cores=1, save_step=True)
dfg = step.orca(
    dfg,
    name="gxtb-OptTS",
    options={"OptTS": None, "NumFreq": None},
    gxtb=True,
    save_step=True,
    use_last_hess=True,
    xtra_inp_str="""%geom\nMaxIter 500\nend""",
)

dfg

[13:25:39] Explicit valence for atom # 11 B, 4, is greater than permitted


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

2026-05-05 13:25:39 INFO  frust.stepper.NONE.job1111: Working dir: .
2026-05-05 13:25:39 INFO  frust.stepper.NONE.job1111: [gxtb-hess] row 0 (ts4)…
2026-05-05 13:26:15 INFO  frust.stepper.NONE.job1111: [gxtb-OptTS] row 0 (ts4)…


,structure_id,custom_name,substrate_name,structure_type,molecule_role,rpos,constraint_atoms,cid,smiles,input_smiles,...,gxtb-hess-EE,gxtb-hess-GE,gxtb-hess-NT,gxtb-hess-input.hess,gxtb-hess-vibs,gxtb-OptTS-EE,gxtb-OptTS-GE,gxtb-OptTS-NT,gxtb-OptTS-oc,gxtb-OptTS-vibs
0,MOL:ts4:structure,ts4,ts4,MOL,structure,<NA>,None,0,None,None,...,-1355.290732,-1354.883088,True,$orca_hessian_file\n\n$act_atom\n 0\n\n$act_c...,"[{'frequency': 87.68}, {'frequency': 108.68}, ...",-1355.29179,-1354.909463,True,"[[-2.181093, -1.515926, 1.006207], [-0.967612,...","[{'frequency': -68.71, 'mode': [[0.000204, -0...."


In [ ]:
from frust.utils.analytics import summarize_ts_vibrations
summarize_ts_vibrations(dfg, "gxtb-OptTS-vibs")
vis.plot_mols(dfg)
vis.plot_vibs(dfg, custom_coords_col_name="gxtb-OptTS-oc")

Found 1 coordinate columns: ['gxtb-OptTS-oc']
Processing 1 rows
Generated 1 molecules for display


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# UMA

In [ ]:
f = Path("../structures/ts2.xyz")
mols = {}
with open(f, "r") as file:
    xyz_block = file.read()
    mol = xyz2mol(xyz_block)
    mols[f.stem] = (mol, [0])

vis.MolTo3DGrid(f, show_charges=True)

step = Stepper(
    step_type="none",
    save_output_dir=True,
    output_base=f"{NAME}_ts2",
    n_cores=10,
    memory_gb=30,
)
dfu = step.build_initial_dfu(mols)
dfu = step.orca(
    dfu,
    name="UMA-OptTS",
    options={"ExtOpt": None, "OptTS": None, "NumFreq": None},
    uma="omol",
    save_step=True,
    xtra_inp_str="""%geom\nMaxIter 500\nend""",
)

dfu

!!! Warning !!! Distance between atoms 13 and 12 (0.785846 A) is suspicious.
!!! Warning !!! Distance between atoms 13 and 12 (0.785846 A) is suspicious.


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

2026-05-05 14:19:44 INFO  frust.stepper.NONE.job1111: Working dir: .
2026-05-05 14:19:47 INFO  frust.stepper.NONE.job1111: [UMA-OptTS] row 0 (ts2)…


,structure_id,custom_name,substrate_name,structure_type,molecule_role,rpos,constraint_atoms,cid,smiles,input_smiles,atoms,coords_embedded,energy_uff,UMA-OptTS-EE,UMA-OptTS-GE,UMA-OptTS-NT,UMA-OptTS-oc,UMA-OptTS-vibs
0,MOL:ts2:structure,ts2,ts2,MOL,structure,<NA>,None,0,None,None,"[H, C, C, C, C, H, C, H, C, H, B, H, H, C, C, ...","[(1.314673, 0.112133, 2.671795), (2.103524, 0....",None,-915.018905,-914.594711,True,"[[1.321448, 0.147911, 2.635206], [2.108974, 0....","[{'frequency': -359.16, 'mode': [[-0.003204, -..."


In [ ]:
summarize_ts_vibrations(dfu, "UMA-OptTS-vibs")
vis.plot_mols(dfu)
vis.plot_vibs(dfu, custom_coords_col_name="UMA-OptTS-oc")

 Structure Ligand RPOS  Status Neg. freqs           Pos. freqs
         0    ts2 <NA> True TS    -359.16 34.3, 45.2, 49.0 ...

Summary:
  True TSs : 1
  Non-TSs  : 0
Found 1 coordinate columns: ['UMA-OptTS-oc']
Processing 1 rows
Generated 1 molecules for display


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

vibs col UMA-OptTS-vibs
Normal mode 0 with frequency -359.16 cm^-1


3Dmol.js failed to load for some reason. Please check your browser console for error messages.